# Your First Mock Server

Mockture is a contract-aware in-process HTTP mock server. Rather than running a real service during tests, you configure Mockture with an OpenAPI contract and a library of named response templates, and it serves those responses from a local port while simultaneously checking every request and response against the schema.

Testing HTTP client code often requires a server to respond. Running a real service introduces network latency, external dependencies, and brittle state. A mock server removes all of that: it runs in the same process, starts in milliseconds, and returns exactly the responses your test needs. Mockture adds contract validation on top, so the mock cannot silently drift out of sync with the real API.

By completing this notebook you will have constructed a Mockture instance, registered a named response, started the server, sent a real HTTP request to it, verified the response shape, and confirmed that the request was recorded.

## Table of Contents

- [Prerequisites](#Prerequisites)
- [1. Setup](#1-Setup)
- [2. The Server Lifecycle](#2-The-Server-Lifecycle)
  - [2.1 Constructing the Server](#21-Constructing-the-Server)
  - [2.2 Registering a Response](#22-Registering-a-Response)
  - [2.3 Starting the Server](#23-Starting-the-Server)
  - [2.4 Making a Request](#24-Making-a-Request)
  - [2.5 Verifying the Call](#25-Verifying-the-Call)
  - [2.6 Inspecting Recorded Requests](#26-Inspecting-Recorded-Requests)
  - [2.7 Stopping the Server](#27-Stopping-the-Server)
- [3. Chaining Interactions](#3-Chaining-Interactions)
  - [3.1 Registering Multiple Responses](#31-Registering-Multiple-Responses)
- [4. Config-Time Validation](#4-Config-Time-Validation)
  - [4.1 Schema Mismatch at Respond Time](#41-Schema-Mismatch-at-Respond-Time)
- [5. Conclusion](#5-Conclusion)

## Prerequisites

No environment variables are required.

Additional prerequisites:

- `mockture` must be installed in the current Python environment.
- `httpx` must be installed (`pip install httpx`).
- `configs/basic_api.openapi.yml` and `configs/basic_api.templates.yml` must be present in the same directory as this notebook.

## 1. Setup

We import `Mockture` from `mockture.server` and build absolute paths to the contract and templates files. Running this cell confirms that both config files are reachable before any server is constructed.

In [ ]:
from pathlib import Path
import httpx
from mockture.server import Mockture

# Paths are relative to this notebook's location.
ROOT = Path(".").resolve()
CONTRACT  = str(ROOT / "configs" / "basic_api.openapi.yml")
TEMPLATES = str(ROOT / "configs" / "basic_api.templates.yml")

print("Contract: ",  CONTRACT)
print("Templates:", TEMPLATES)

## 2. The Server Lifecycle

Mockture's lifecycle has four phases: construct, register, start, and stop. Each phase has a clear boundary. Operations in earlier phases carry forward—a response registered before `start()` is still served after the server is running.

### 2.1 Constructing the Server

We construct a `Mockture` instance with the contract path, templates path, and `strict=True`. Constructing the server loads both files into memory but does not bind to any port.

- **contract**: the OpenAPI YAML or JSON file that defines the expected request and response schemas for every endpoint.
- **strict mode**: when `True`, a request or response that violates the contract schema returns HTTP 500 instead of the configured response. When `False`, the violation is recorded but the response is served normally.

In [ ]:
mock = Mockture(
    contract_path=CONTRACT,
    templates_path=TEMPLATES,
    strict=True,
    port=0,  # 0 = pick a random free port
)
print("Mockture instance created (server not started yet)")

### 2.2 Registering a Response

`respond()` registers what the server should return when a matching request arrives. It looks up the named template, resolves args (kwargs override template defaults), renders the response body, and validates it against the OpenAPI response schema immediately.

- **template**: a named entry in the templates YAML file specifying the HTTP method, path, and response for one scenario.
- **interaction**: a template instance with all args fully resolved, ready to be served.

If the rendered body does not conform to the schema, `ContractConfigError` is raised at this point—before any server starts. Schema mismatches are caught at configuration time, not mid-test.

In [ ]:
mock.respond(
    "create_order_success",
    order_id="ord-notebook-1",
    order_status="queued",
    # status_code is not overridden — keeps the template default of 201
)
print("Interaction registered. No server running yet.")

### 2.3 Starting the Server

`start()` binds the server to the configured port and begins serving requests. After this call, `mock.base_url` and `mock.url_for(path)` return usable URLs.

In [ ]:
mock.start()
print("Server listening at:", mock.base_url)
print("POST /orders URL:   ", mock.url_for("/orders"))

### 2.4 Making a Request

We send a well-formed `POST /orders` request using `httpx`. The mock receives the request, validates it against the `CreateOrderRequest` schema, and returns the response registered in section 2.2. The response body is exactly what was configured in `respond()`—the `order_id` and `status` values come from the args we passed.

In [ ]:
response = httpx.post(
    mock.url_for("/orders"),
    json={"item_id": "SKU-1", "quantity": 2},
    timeout=5.0,
)

print("Status:", response.status_code)
print("Body:  ", response.json())

### 2.5 Verifying the Call

`assert_called()` raises `AssertionError` if the endpoint was not called exactly the expected number of times. `assert_no_contract_violations()` raises if any request or response violated the schema during this test.

In [ ]:
mock.assert_called(path="/orders", method="POST", times=1)
mock.assert_no_contract_violations()
print("All assertions passed.")

### 2.6 Inspecting Recorded Requests

`calls_for()` returns a `CallView` with all recorded requests for a given path and method. Each record exposes the request body, headers, and response status code.

- **CallView**: a filtered view of the call log for a specific endpoint, exposing `count` and `records`.
- **CallRecord**: a single recorded interaction containing `json_body`, `headers`, `method`, `path`, and `status_code`.

In [ ]:
view = mock.calls_for("/orders", "POST")
print("Call count:", view.count)
for record in view.records:
    print("  Request body:   ", record.json_body)
    print("  Response status:", record.status_code)

### 2.7 Stopping the Server

`stop()` shuts down the server and resets all registered interactions and recorded calls. Every test should stop its server—failing to do so leaves the port occupied for the duration of the process.

In [ ]:
mock.stop()
print("Server stopped.")

## 3. Chaining Interactions

`respond()` returns `self`, making it possible to register multiple interactions in a single expression. This is useful when a test exercises a sequence of endpoints that must all be available before the first request.

### 3.1 Registering Multiple Responses

We construct a fresh instance and chain two `respond()` calls—one for `POST /orders` and one for `GET /orders/{order_id}`—before calling `start()`.

In [ ]:
mock2 = Mockture(contract_path=CONTRACT, templates_path=TEMPLATES, strict=True)

(
    mock2
    .respond("create_order_success", order_id="ord-chain", order_status="created")
    .respond("get_order",            order_id="ord-chain", order_status="created")
)

mock2.start()

r1 = httpx.post(mock2.url_for("/orders"),          json={"item_id": "A", "quantity": 1}, timeout=5.0)
r2 = httpx.get( mock2.url_for("/orders/ord-chain"),                                      timeout=5.0)

print("POST status:", r1.status_code, "|", r1.json())
print("GET  status:", r2.status_code, "|", r2.json())

mock2.stop()

## 4. Config-Time Validation

Contract validation begins the moment `respond()` is called—not when the server starts, and not when a request arrives. This section demonstrates what happens when a template's body does not match the schema.

### 4.1 Schema Mismatch at Respond Time

The `invalid_success_shape` template in `configs/basic_api.templates.yml` intentionally returns `{"bad_field": "should_fail"}` as a 201 body—a body that does not match the `OrderResponse` schema. With `strict=True`, calling `respond()` with this template raises `ContractConfigError` immediately. The server is never started.

- **ContractConfigError**: raised when a template's rendered response body does not conform to the OpenAPI response schema for that method, path, and status code.

In [ ]:
from mockture.errors import ContractConfigError

mock3 = Mockture(contract_path=CONTRACT, templates_path=TEMPLATES, strict=True)

try:
    mock3.respond("invalid_success_shape")
    print("ERROR: expected ContractConfigError but nothing was raised")
except ContractConfigError as e:
    print("ContractConfigError raised as expected:")
    print(" ", e)

## 5. Conclusion

This notebook covered the complete Mockture lifecycle:

- Constructed a `Mockture` instance against an OpenAPI contract and a templates file.
- Registered a named response with `respond()`, confirming config-time schema validation runs immediately.
- Started the server and sent a real HTTP request to it.
- Verified the call count with `assert_called()` and schema compliance with `assert_no_contract_violations()`.
- Inspected the recorded call log with `calls_for()`.
- Chained multiple `respond()` calls on a single instance.
- Observed `ContractConfigError` raised at `respond()` time for a schema-violating template.

The `02_plugin_explicit` and `03_plugin_ini` examples show how the pytest plugin automates the `start()` and `stop()` lifecycle so test functions contain only HTTP calls and assertions. For a detailed look at what strict mode does at request time, see the `06_strict_modes` tutorial notebook.